In [ ]:
from paths import *
import os
import numpy as np
from scipy.stats import ttest_rel, wilcoxon

In [ ]:
# Configs
envS_path = os.path.join(model_save_, f"eval-C_0Tj-1021123825")
envP_path = os.path.join(model_save_, f"eval-C_0Tk-1021183234")
task_name = "ABXSomething-stop-data-aspirationRangeComp"
range_start = 0
range_end = 2
model_condition = "b"
strseq_learned_runs = "12345"

epoch_range = (90, 100)


In [ ]:
# Collect data
envS_res = {}
envP_res = {}
for hiddim in [3, 8, 16, 32, 48, 64]: 
    model_type = f"recon{hiddim}-phi"
    for environment, env_res, res_save_dir in zip(["S", "P"], [envS_res, envP_res], [envS_path, envP_path]):
        layered_res = {}
        look_for_layer_path = f"{task_name}-{range_start}-{range_end}"
        # Read ori
        ori_path = os.path.join(res_save_dir, look_for_layer_path, f"07-save-ari-recon64-phi-{model_condition}-{strseq_learned_runs}-ori.npy")
        if os.path.exists(ori_path): 
            ori_res = np.load(ori_path)
        else: 
            raise ValueError("No ori path found.")
        
        for layer in ["hidrep", "attnout", "dec-lin1", "enc-lin1", 
                        "dec-rnn1-f", "enc-rnn1-f", "enc-rnn1-b",
                        "dec-rnn2-f", "enc-rnn2-f", "enc-rnn2-b", 
                        "dec-rnn3-f", "enc-rnn3-f", "enc-rnn3-b", 
                        "dec-rnn4-f", "enc-rnn4-f", "enc-rnn4-b", 
                        "dec-rnn5-f", "enc-rnn5-f", "enc-rnn5-b", ]: # "enc-lin1", 
            print(f"Processing {model_type} in layer {layer}...")
            asp_list_epochs = []
            layer_path = os.path.join(res_save_dir, look_for_layer_path, 
                                        f"07-save-ari-{model_type}-{model_condition}-{strseq_learned_runs}-{layer}.npy")
            if os.path.exists(layer_path): 
                layer_res = np.load(layer_path)
            else: 
                print(f"Warning: {layer_path} not found. ")
                layer_res = np.zeros_like(ori_res)
            layered_res[layer] = layer_res
        layered_res["ori"] = ori_res
        env_res[hiddim] = layered_res


In [ ]:
# Define test functions
def test_arrays(data1, data2, test_epoch_range=(0, 101)): 
    # Assuming `data1` and `data2` are your two numpy arrays of shape (num_runs, num_epochs)
    # Choose your test (paired t-test or Wilcoxon signed-rank test)
    p_values = {}
    t_stats = {}

    # Loop over each epoch to perform the significance test
    for epoch in test_epoch_range:
        t_stats[epoch], p_values[epoch] = ttest_rel(data1[:, epoch], data2[:, epoch])
    return t_stats, p_values

def test_arrays_merged(data1, data2, test_epoch_range=(0, 101)): 
    # Assuming `data1` and `data2` are your two numpy arrays of shape (num_runs, num_epochs)
    # Choose your test (paired t-test or Wilcoxon signed-rank test)

    data1_pooled = data1[:, test_epoch_range[0]:test_epoch_range[1]].flatten()
    data2_pooled = data2[:, test_epoch_range[0]:test_epoch_range[1]].flatten()

    return ttest_rel(data1_pooled, data2_pooled)

In [ ]:
# Shape of each layer's result: (num_runs, num_epochs)
test_hiddim = 32
test_epoch_range = (90, 101)
envS_ori_attn_diff = envS_res[test_hiddim]["attnout"] - envS_res[test_hiddim]["ori"]
envP_ori_attn_diff = envP_res[test_hiddim]["attnout"] - envP_res[test_hiddim]["ori"]
# Shape: (num_runs, num_epochs)

t_stats, p_values = test_arrays(envS_ori_attn_diff, envP_ori_attn_diff, test_epoch_range)
merged_t_stat, merged_p_value = test_arrays_merged(envS_ori_attn_diff, envP_ori_attn_diff, test_epoch_range)
